# 流式调用（stream）

举例


In [4]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model


#读取.env配置文件中的信息。
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv('DEEPSEEK_API_KEY')
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

#模型的初始化
model = init_chat_model(
    api_base=DEEPSEEK_BASE_URL,
    base_url=DEEPSEEK_BASE_URL,
    model='deepseek:deepseek-v4-flash',
)

In [5]:
for chunk in model.stream('帮我解释一下什么是人工智能'):
    print(chunk.text,end='',flush=True) #逐token输出

简单说：**人工智能（AI）就是让机器表现出类似人类智能行为的技术和科学**。它不是某一种具体技术，而是一大类方法的总称，目标是让计算机能够感知、学习、推理、决策、理解语言、识别图像，甚至创作内容。

可以拆成几个层面来理解：

### 1. AI 想解决什么问题？
人类做很多事需要“智能”：看东西、听懂话、做判断、下棋、开车、写文章。AI 就是试图让机器也能完成这些任务。比如：
- 人脸识别：让机器“看懂”图像；
- 语音助手：让机器“听懂”并回应；
- 推荐系统：猜你可能喜欢什么视频或商品；
- 自动驾驶：感知路况并做决策；
- ChatGPT 这类大模型：根据上下文生成文字、回答问题。

### 2. AI 的核心思路是什么？
现代 AI 最主流的方法是**机器学习**，尤其是**深度学习**。传统程序靠人写规则：如果这样，就那样。机器学习则不同：给机器大量数据，让它自己找规律。

比如教机器认猫，不是告诉它“猫有胡须、尖耳朵、尾巴”，而是给它看几百万张猫的图片，让它自己学会猫的特征。大语言模型也是类似：它读了海量文本，学会词语之间的统计规律，然后根据你的问题预测接下来最可能出现的词，从而生成回答。

所以可以打个比方：**AI 像一个超级模式识别器，擅长从大量例子中总结规律，并用来完成新任务。**

### 3. AI 有不同类型
按能力范围分：
- **弱人工智能 / 窄 AI**：只擅长特定任务，比如下围棋、识别人脸、翻译语言。目前我们用的几乎都是这一类。
- **强人工智能 / 通用 AI（AGI）**：像人一样能处理各种任务，有通用推理能力。目前还没有实现。
- **超人工智能**：理论上全面超越人类，目前属于科幻和理论讨论。

按技术路线分：
- **符号主义**：用逻辑、规则、知识库模拟智能；
- **连接主义**：用神经网络模拟大脑连接；
- **行为主义**：强调感知和行动，比如机器人通过环境反馈学习。

### 4. AI 不等于什么？
很多人会误解：
- **AI 不等于机器人**。机器人只是 AI 可能运行的载体，AI 也可以只是软件。
- **AI 不等于有意识**。它可能表现得很聪明，但通常没有自我意识、情感和真正理解。
- **AI 不是全知全能**。它会犯错、会产生“幻觉”、会受训练数据偏见影响。
- **AI 不一定是人形的*

# 2.批量调用（batch）

举例1：一次性接受所有响应

In [6]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]

responses = model.batch(messages)

for response in responses:
    print(response.content)

你好！我是 DeepSeek，一个由深度求索公司开发的 AI 助手。我可以帮你回答问题、写作、编程、翻译、总结信息等。有什么我可以帮你的吗？
17
中国首都是**北京**。


举例2：按照完成的顺序接收响应

In [7]:
messages = [
    "你好，你是谁？",
    "2 + 3 * 5 = ?",
    "中国首都在哪里？"
]

responses = model.batch_as_completed(messages)

for response in responses:
    print(response)

(1, AIMessage(content='17', additional_kwargs={'refusal': None, 'reasoning_content': 'We need answer simple order ops. 2 + 3*5 = 2+15=17. Need final only maybe.'}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 39, 'total_tokens': 68, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 27, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 39}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'fdfdc788-572c-4ff3-9c12-a6ef598f63c8', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a1269b-8719-7c10-8432-46c7551b2ea9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 39, 'output_tokens': 29, 'total_tokens': 68, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasonin

举例3：性能对比

使用batch()

In [8]:
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

# 批量调用（高效）
import time

start = time.time()
responses = model.batch(inputs)
batch_time = time.time() - start

print("批量调用结果：")
for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")
print(f"耗时: {batch_time:.2f}秒\n")

批量调用结果：
1. Spring has arrived.
2. Summer is very hot.
3. Autumn fallen leaves
4. It snows in winter.
耗时: 1.29秒



作为对比，演示循环调用invoke（）

In [12]:
# 循环调用（低效，仅用于对比）
inputs = [
    "翻译成英文：春天来了",
    "翻译成英文：夏天很热",
    "翻译成英文：秋天落叶",
    "翻译成英文：冬天下雪"
]

start = time.time()
loop_responses = []
for inp in inputs:
    response = model.invoke(inp)
    loop_responses.append(response)
loop_time = time.time() - start

for i, response in enumerate(responses):
    print(f"{i+1}. {response.content}")

print(f"循环调用耗时: {loop_time:.2f}秒")

print(f"批量调用节省: {((loop_time - batch_time) / loop_time * 100):.1f}%")

1. Spring has come.
2. Summer is very hot.
3. Autumn leaves
4. It snows in winter.
循环调用耗时: 4.66秒
批量调用节省: 67.0%
